In [8]:
import pandas as pd
import numpy as np
import joblib


In [9]:
import joblib
model = joblib.load("recovery_model.pkl")

In [12]:
df = pd.read_csv("synthetic_cases.csv")


In [13]:
feature_cols = [
    "amount_due",
    "days_overdue",
    "past_defaults",
    "credit_score",
    "dca_success_rate",
    "contact_attempts"
]

X = df[feature_cols]


In [14]:
df["predicted_recovery_probability"] = model.predict_proba(X)[:, 1]


In [15]:
df["risk_score"] = 1 - df["predicted_recovery_probability"]


In [16]:
def assign_priority(risk):
    if risk > 0.6:
        return "HIGH"
    elif risk >= 0.3:
        return "MEDIUM"
    else:
        return "LOW"

df["priority"] = df["risk_score"].apply(assign_priority)


In [17]:
df["status"] = df["recovered"].map({
    1: "RECOVERED",
    0: "IN_PROGRESS"
})


In [18]:
dca_list = ["DCA_01", "DCA_02"]
df["dca_id"] = np.random.choice(dca_list, size=len(df))


In [19]:
output_df = df[
    [
        "case_id",
        "amount_due",
        "days_overdue",
        "credit_score",
        "dca_id",
        "dca_success_rate",
        "risk_score",
        "priority",
        "status"
    ]
]


In [20]:
output_df.to_csv("dashboard_predictions.csv", index=False)


In [21]:
print(output_df.head())
print(output_df["priority"].value_counts())


  case_id  amount_due  days_overdue  credit_score  dca_id  dca_success_rate  \
0      C1       16795           176           368  DCA_01          0.625513   
1      C2        1860           358           601  DCA_02          0.410045   
2      C3       77820           253           452  DCA_02          0.746749   
3      C4       55886           246           836  DCA_01          0.841640   
4      C5        7265           227           334  DCA_01          0.888430   

   risk_score priority     status  
0    0.498255   MEDIUM  RECOVERED  
1    0.570304   MEDIUM  RECOVERED  
2    0.466025   MEDIUM  RECOVERED  
3    0.330020   MEDIUM  RECOVERED  
4    0.525146   MEDIUM  RECOVERED  
priority
MEDIUM    7566
LOW       1851
HIGH       583
Name: count, dtype: int64


In [22]:
output_df.groupby("priority")["risk_score"].mean()


priority
HIGH      0.636456
LOW       0.252141
MEDIUM    0.431973
Name: risk_score, dtype: float64

In [23]:
output_df.groupby("priority")["risk_score"].agg(["min", "mean", "max"])


,min,mean,max
priority,,,
HIGH,0.600019,0.636456,0.718370
LOW,0.134003,0.252141,0.299995
MEDIUM,0.300035,0.431973,0.599827
